# Tutorial: Dynamic Multi-Agent Systems in Vinagent

Welcome to the **Dynamic Multi-Agent System** tutorial! In this guide, we will explore how to build, orchestrate, and execute workflows where an AI Planner dynamically determines the execution graph—assigning tasks, mapping dependencies, and running operations in parallel.

## 1. Introduction: Solution Architecture

The **Dynamic Multi-Agent** architecture consists of three primary phases:
1. **Planning**: A dedicated _Planner Agent_ (using structured LLM outputs) receives the user query. It dynamically plans a sequence of steps, creating a `TaskGraph`.
2. **Execution**: The orchestrator (`DynamicCrew`) processes the `TaskGraph`. Using Kahn's topological sort, it executes independent tasks in **parallel** and dependent tasks sequentially.
3. **Aggregation**: Once all specialist agents have completed their tasks, an _Aggregator Agent_ synthesizes the intermediate results into a final cohesive output.

## 2. Why Does It Matter?

**Comparing Static vs. Dynamic Multi-Agent Systems**

- **Static Multi-Agent**: Workflows are explicitly defined in code (e.g., using a fixed `StateGraph`). The flow (A → B → C) never changes. It is predictable but brittle when facing novel queries that require customized approaches.
- **Dynamic Multi-Agent**: The graph is built at runtime! The LLM evaluates the specific request and generates the optimal execution topology. This allows the system to scale its complexity gracefully. Additionally, by extracting dependency mapping, it natively achieves **Parallel Processing**, drastically reducing latency for independent operations.

## 3. Setup

First, install the required dependencies (if you haven't already) and ensure your `.env` contains your LLM API Key.

In [ ]:
# !pip install python-dotenv langchain-openai langchain-core pydantic

import os
from dotenv import load_dotenv, find_dotenv

# Ensure you have your OPENAI_API_KEY in the .env file
load_dotenv(find_dotenv('.env'))

## 4. Building a Dynamic Financial Multi-Agent System

Let's build a team of specialist agents designed to perform comprehensive stock market analyses. We will configure agents for Data Crawling, Legal Assessment, and Hypothesis Generation, and finally wrap them in a `DynamicCrew`.

In [ ]:
from langchain_openai import ChatOpenAI
from vinagent.agent.agent import Agent
from vinagent.multi_agent.dynamic_crew import DynamicCrew

llm = ChatOpenAI(model="gpt-4o")

# 1. Define Specialist Agents
# Define the specialized agents
agents = {
    "Data Crawler Agent": Agent(
        description="You crawl, transform, and normalize market data for modeling.",
        llm=llm,
        tools=[
            "vinagent/tools/yfinance_tools.py",
            "vinagent/tools/vnstock_tools.py",
            "vinagent/tools/websearch_tools.py"
        ],
        memory_path="vinagent/templates/finance_group/data_crawler/memory.json",
        tools_path = "vinagent/templates/finance_group/data_crawler/tool.json"
    ),
    "Data Analyst": Agent(
        description="You analyze data to get statistical descriptions from the Data Crawler Agent.",
        llm=llm,
        tools=[],
        memory_path="vinagent/templates/finance_group/data_analyst/memory.json",
        tools_path = "vinagent/templates/finance_group/data_analyst/tool.json"
    ),
    "Legal Assistant": Agent(
        description="You analyze the legal risks in the portfolio.",
        llm=llm,
        tools=[],
        memory_path="vinagent/templates/finance_group/legal_assistant/memory.json",
        tools_path = "vinagent/templates/finance_group/legal_assistant/tool.json"
    ),
    "Searcher Agent": Agent(
        description="You gather financial data and market news using web search tools.",
        llm=llm,
        tools=[
            "vinagent/tools/websearch_tools.py"
        ],
        memory_path="vinagent/templates/finance_group/researcher/memory.json",
        tools_path = "vinagent/templates/finance_group/researcher/tool.json"
    ),
    "Hypothesis Agent": Agent(
        description="You analyze the gathered data and formulate investment hypotheses.",
        llm=llm,
        tools=[],
        memory_path="vinagent/templates/finance_group/hypothesis_maker/memory.json",
        tools_path = "vinagent/templates/finance_group/hypothesis_maker/tool.json"
    ),
    "Signal Builder Agent": Agent(
        description="You convert hypotheses into actionable trading signals with confidence scores.",
        llm=llm,
        tools=[],
        memory_path="vinagent/templates/finance_group/signal_builder/memory.json",
        tools_path = "vinagent/templates/finance_group/signal_builder/tool.json"
    ),
    "Reporter Agent": Agent(
        description="You generate a comprehensive visualization report with charts and recommendations.",
        llm=llm,
        tools=[],
        memory_path="vinagent/templates/finance_group/reporter/memory.json",
        tools_path = "vinagent/templates/finance_group/reporter/tool.json"
    ),
}

# Optional: A dedicated aggregator, or we can just let DynamicCrew use the base LLM
aggregator = Agent(
    description="You synthesize all analytical outputs into a final executive summary.",
    llm=llm,
    tools=[],
    memory_path="vinagent/templates/finance_group/aggregator/memory.json",
    tools_path = "vinagent/templates/finance_group/aggregator/tool.json"
)

# 2. Initialize DynamicCrew Orchestrator
crew = DynamicCrew(
    llm=llm,
    agents=agents
)

crew

### Orchestrating the Execution

Now, we pass a complex query to the `DynamicCrew`. We use `ainvoke()` to execute the plan asynchronously, enabling topological groupings to process **in parallel**.

In [ ]:
import asyncio

query = (
    "Perform a comprehensive stock market analysis on Apple Inc. (AAPL). "
    "Crawl the latest data, assess legal risks, and build predictive models "
    "before returning a final recommendation."
)

async def run_analysis():
    print(f"Submitting Query: {query}\n")
    
    # Execute the dynamically generated plan
    # Under the hood, this will generate a TaskGraph and run independent steps simultaneously!
    result = await crew.ainvoke(query)
    
    print("\n--- Final Aggregated Result ---")
    print(result)

# Run the async loop
await run_analysis()

## 5. Conclusion

In this tutorial, we demonstrated the power of **Dynamic Multi-Agent Systems**:
1. **Flexibility**: We did not tie our layout down to a pre-coded graph. The Planner intelligently identified how to structure the problem specifically to answer the query.
2. **Performance**: Tasks that shared no dependencies were delegated to their respective specialist agents to execute concurrently.
3. **Extensibility**: To add capabilities, a developer only needs to register a new `Agent` with its descriptions into the `DynamicCrew`'s agent dictionary. The Planner will naturally begin to incorporate it when relevant.